In [1]:
!ls /kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage
#!ls /kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/train

train  val


In [2]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

train_dir = "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/train"
test_dir = "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/val"

BATCH_SIZE = 64
IMG_SIZE = (224,224)

train_ds = image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)


test_ds = image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)


print("Oto jedyna, matematycznie prawdziwa lista klas Twojego datasetu:\n")
print("class_names = [")
for name in train_ds.class_names:
    print(f'    "{name}",')
print("]")

2026-06-06 20:39:29.489869: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780778369.661284      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780778369.712914      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780778370.118807      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780778370.118852      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780778370.118854      23 computation_placer.cc:177] computation placer alr

Found 43444 files belonging to 38 classes.


I0000 00:00:1780778409.145434      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 10861 files belonging to 38 classes.
Oto jedyna, matematycznie prawdziwa lista klas Twojego datasetu:

class_names = [
    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___Cedar_apple_rust",
    "Apple___healthy",
    "Blueberry___healthy",
    "Cherry_(including_sour)___Powdery_mildew",
    "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)",
    "Peach___Bacterial_spot",
    "Peach___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Raspberry___healthy",
    "Soybean___healthy",
    "Squash___Powdery_mildew",
    "Strawberry___Lea

In [3]:
from tensorflow.keras import layers, models

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    
    layers.RandomFlip("horizontal"),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    
    base_model,
    
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(38, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_brightness               │ (None, 224, 224, 3)    │             0 │
│ (RandomBrightness)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ (None, 224, 224, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │         4,902 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,426,854 (9.26 MB)

 Trainable params: 168,870 (659.65 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [4]:
print("--starting first stage--")

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5
)

--starting first stage--
Epoch 1/5


I0000 00:00:1780778419.804376      73 service.cc:152] XLA service 0x7ba78c211220 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780778419.804415      73 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1780778421.011757      73 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1780778429.202575      73 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


679/679 ━━━━━━━━━━━━━━━━━━━━ 106s 134ms/step - accuracy: 0.7413 - loss: 0.9920 - val_accuracy: 0.9390 - val_loss: 0.1967
Epoch 2/5
679/679 ━━━━━━━━━━━━━━━━━━━━ 32s 47ms/step - accuracy: 0.9292 - loss: 0.2232 - val_accuracy: 0.9556 - val_loss: 0.1433
Epoch 3/5
679/679 ━━━━━━━━━━━━━━━━━━━━ 32s 47ms/step - accuracy: 0.9467 - loss: 0.1663 - val_accuracy: 0.9591 - val_loss: 0.1333
Epoch 4/5
679/679 ━━━━━━━━━━━━━━━━━━━━ 32s 47ms/step - accuracy: 0.9538 - loss: 0.1403 - val_accuracy: 0.9582 - val_loss: 0.1288
Epoch 5/5
679/679 ━━━━━━━━━━━━━━━━━━━━ 32s 47ms/step - accuracy: 0.9595 - loss: 0.1246 - val_accuracy: 0.9634 - val_loss: 0.1112


In [5]:
print("--preparing model for second stage--")

unfreezeModel = model.layers[4]
unfreezeModel.trainable = True

for layer in unfreezeModel.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

--preparing model for second stage--


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_brightness               │ (None, 224, 224, 3)    │             0 │
│ (RandomBrightness)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ (None, 224, 224, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │         4,902 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,426,854 (9.26 MB)

 Trainable params: 1,695,270 (6.47 MB)

 Non-trainable params: 731,584 (2.79 MB)

In [6]:
print("--starting second stage - fine-tuning--")

history_f = model.fit(
    train_ds,
    validation_data = test_ds,
    epochs=7
)

--starting second stage - fine-tuning--
Epoch 1/7


2026-06-06 20:44:21.186650: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-06 20:44:21.385080: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


678/679 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.8301 - loss: 0.5835

2026-06-06 20:44:59.092116: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-06 20:44:59.292826: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


679/679 ━━━━━━━━━━━━━━━━━━━━ 69s 77ms/step - accuracy: 0.8302 - loss: 0.5829 - val_accuracy: 0.9595 - val_loss: 0.1159
Epoch 2/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 35s 52ms/step - accuracy: 0.9474 - loss: 0.1645 - val_accuracy: 0.9643 - val_loss: 0.0983
Epoch 3/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 35s 52ms/step - accuracy: 0.9592 - loss: 0.1210 - val_accuracy: 0.9699 - val_loss: 0.0854
Epoch 4/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 35s 52ms/step - accuracy: 0.9684 - loss: 0.0962 - val_accuracy: 0.9734 - val_loss: 0.0769
Epoch 5/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 35s 52ms/step - accuracy: 0.9750 - loss: 0.0768 - val_accuracy: 0.9766 - val_loss: 0.0711
Epoch 6/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 35s 52ms/step - accuracy: 0.9782 - loss: 0.0649 - val_accuracy: 0.9783 - val_loss: 0.0650
Epoch 7/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 35s 52ms/step - accuracy: 0.9829 - loss: 0.0536 - val_accuracy: 0.9796 - val_loss: 0.0609


In [7]:
# Zapisanie wyszkolonego modelu 
model_save_path = '/kaggle/working/plant_disease_detector_v2.keras'
model.save(model_save_path)

print(f"Model został pomyślnie zapisany w: {model_save_path}")

Model został pomyślnie zapisany w: /kaggle/working/plant_disease_detector_v2.keras
